# 04 — Error analysis: which photos does the model get wrong, and why?

Notebook 03 told us **how often** a model is wrong. This notebook shows **which photos** it got wrong
and **where on the face it was looking** when it made the mistake.

There are two kinds of mistake (fake = 1 is the thing we want to catch):

| Mistake | What happened | Why it matters in KYC |
|---|---|---|
| **False accept** | a **fake** face the model called real | fraud gets through (the `far` score) |
| **False reject** | a **real** customer the model called fake | a genuine person is blocked (the `frr` score) |

For the worst mistakes we draw a **Grad-CAM heatmap**: red areas are the parts of the face that pushed the model
towards "fake". If the red is on the background or the image edges instead of the face, the model may have learned
a shortcut instead of real signs of AI generation.

**How to use:** this notebook is made for **Kaggle**. Do the settings below once, then run the cells from top to
bottom (Shift+Enter).

### Before you start: Kaggle settings

Open the right-hand panel of the notebook editor (the **>|** icon at the top right if it is hidden):

- **Settings → Internet → On.** Step 1 downloads the project code from GitHub. Kaggle asks you to verify your phone once.
- **Add Input → Datasets →** search **"140k Real and Fake Faces"** (by xhlulu) → **+**. These are the images.
  Kaggle attaches them read-only under `/kaggle/input/`, so nothing is downloaded or unzipped.
- **Add Input → Your Work → Notebooks →** your saved notebook 05 → **+**. This brings in its trained models
  (`outputs/<experiment>/best.pt`) and score files, read-only. Notebook 05 must be saved first with
  **Save Version → Save & Run All (Commit)**.
- **Settings → Accelerator → GPU T4 x2.** Makes the model steps much faster (free, about 30 GPU hours a week).

After you change a setting, Kaggle restarts the session: run the cells again from the top.

## Step 1 — Get the project, the tools and the trained models

**What this does, in plain words:**

1. Loads the Python tools we need.
2. Downloads the project code (`src/`, `configs/`) from GitHub into `/kaggle/working/`, the only folder on Kaggle
   where we can write files. If Internet is off, it tells you so in plain words.
3. Finds the dataset you attached with **Add Input** somewhere under `/kaggle/input/`.
4. Installs `grad-cam` (the heatmap library) if it is missing. Kaggle already has PyTorch and `timm`.
5. Lists the trained models (`best.pt`) in the notebooks you attached with **Add Input → Your Work**
   and in this notebook's own `outputs/`, and counts the test photos.

In [ ]:
# 1. Tools we need
import importlib.util
import os
import random
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# 2. Get the project code (src/ and configs/) from GitHub.
#    It goes into /kaggle/working: the only folder on Kaggle where we can write files.
REPO_URL = "https://github.com/msondhi/UOT_SCS_3546_041_Deep-Learning.git"
PROJECT_ROOT = Path("/kaggle/working/UOT_SCS_3546_041_Deep-Learning")

if not PROJECT_ROOT.exists():
    result = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_ROOT)])
    if result.returncode != 0:
        raise RuntimeError("Could not download the project from GitHub. "
                           "Turn on Settings -> Internet in the right-hand panel, then run this cell again.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))   # lets Python import the project's own code
print("Project folder:", PROJECT_ROOT)

# 3. Find the dataset you attached with Add Input. Kaggle puts it somewhere under /kaggle/input.
def find_dataset():
    """Look inside /kaggle/input for the folder called real-vs-fake that has train/ valid/ test/."""
    for depth in range(1, 5):                                  # look 1, 2, 3, then 4 folders deep
        pattern = "/".join(["*"] * depth) + "/real-vs-fake"    # e.g. "*/*/real-vs-fake"
        for folder in Path("/kaggle/input").glob(pattern):
            if (folder / "train").is_dir():
                return folder
    return None

DATA_ROOT = find_dataset()       # folder with train/ valid/ test/, or None if not attached
if DATA_ROOT is None:
    print('Dataset not found: Add Input -> Datasets -> "140k Real and Fake Faces", then run this cell again.')
else:
    print("Images folder:", DATA_ROOT)

# 4. Install any missing package (Kaggle already has PyTorch, so it is never reinstalled).
# {name used in "import": name used by pip}
needed = {"timm": "timm", "pytorch_grad_cam": "grad-cam"}
for module, package in needed.items():
    if importlib.util.find_spec(module) is None:
        print("Installing", package, "...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

# 5. Folders to search for trained models and scores:
#  - outputs/ of other notebooks you attached with Add Input -> Your Work (read-only),
#    e.g. your saved notebook 05: /kaggle/input/<notebook>/.../outputs/
#  - this notebook's own outputs/ in /kaggle/working (listed last, so a new file made here wins)
RESULTS_FOLDERS = []
for depth in range(1, 5):
    RESULTS_FOLDERS += sorted(Path("/kaggle/input").glob("/".join(["*"] * depth) + "/outputs"))
RESULTS_FOLDERS.append(PROJECT_ROOT / "outputs")

checkpoints = []
for results_folder in RESULTS_FOLDERS:
    checkpoints += sorted(results_folder.glob("*/best.pt"))
print(f"Trained models found: {len(checkpoints)}")
for file in checkpoints:
    print("  ", file.parent.name, "  ", file)
if not checkpoints:
    print("Attach your saved notebook 05 (Add Input -> Your Work -> Notebooks), then run this cell again.")

count = lambda folder: len(os.listdir(folder)) if folder.is_dir() else 0
TEST_DIR = DATA_ROOT / "test" if DATA_ROOT is not None else Path("missing")
print(f"Test photos: real={count(TEST_DIR / 'real'):,}  fake={count(TEST_DIR / 'fake'):,}")

## Step 2 — Pick one model and load it

**What this does:** loads one trained model. The settings it was trained with (model type, image size, …) are saved
inside `best.pt`, so the model is rebuilt exactly as it was trained. It also tells you how the model was trained:
on clean + randomly damaged photos (the project default) or on clean photos only (the `06_effnet_clean` control).

**Choose the model:** `MODEL = None` uses the first one listed in Step 1. To pick one yourself, write its
experiment name, e.g. `MODEL = "03_effnet_finetune"`.

In [ ]:
import torch

from deepfake_kyc.models import load_checkpoint
from deepfake_kyc.utils import get_device

MODEL = None      # None = the first model found in Step 1; or e.g. "03_effnet_finetune"

matches = [path for path in checkpoints if MODEL is None or path.parent.name == MODEL]
if not matches:
    raise FileNotFoundError(f"No trained model called {MODEL!r}. Choose one of: {[p.parent.name for p in checkpoints]}")
CHECKPOINT = matches[0]

device = get_device()                                   # GPU if there is one, otherwise CPU
model, settings = load_checkpoint(CHECKPOINT, device)   # the model + the settings saved inside it
IMAGE_SIZE = settings["data"]["image_size"]

trained_with_damage = settings["augment"]["degradation"]
print("Model:     ", CHECKPOINT.parent.name, f"({settings['model']['name']})")
print("Trained on:", "clean + randomly damaged photos" if trained_with_damage else "clean photos only")
print("Image size:", IMAGE_SIZE, "x", IMAGE_SIZE)
print("Running on:", device)

## Step 3 — Run the model on the test photos

**What this does:** shows the model the test photos (real and fake mixed). For each photo the model gives a number
between 0 and 1, `P(fake)`: how sure it is that the face is AI-generated. If `P(fake)` is 0.5 or more we call it "fake".

**Controls:**
- `TEST_PHOTOS = 2000` uses 2,000 random test photos so it finishes quickly; `None` = all 20,000.
  With the same number, these are **the same photos** that notebook 03 (Step 1b) scored, so the numbers match.
- `THRESHOLD = 0.5`: the cut-off between "real" and "fake".

In [ ]:
from torch.utils.data import DataLoader

from deepfake_kyc.data import build_transform, make_dataset, subsample
from deepfake_kyc.engine import predict

TEST_PHOTOS = 2000     # None = all test photos
THRESHOLD = 0.5

# The same image preparation the model was tested with (resize, no random changes).
transform = build_transform(IMAGE_SIZE)
all_test_photos = make_dataset(TEST_DIR, transform=transform)
test_photos = subsample(all_test_photos, TEST_PHOTOS, settings["seed"])   # same pick as evaluate.py

# File path of every photo we use, in the same order as the predictions.
if test_photos is all_test_photos:
    paths = [path for path, _ in all_test_photos.samples]
else:
    paths = [all_test_photos.samples[i][0] for i in test_photos.indices]

# num_workers = how many CPU cores prepare photos in parallel (Kaggle has 4), so the GPU never waits.
loader = DataLoader(test_photos, batch_size=64, shuffle=False, num_workers=os.cpu_count(),
                    pin_memory=torch.cuda.is_available())
probs, labels, _ = predict(model, loader, device)   # probs = P(fake) per photo, labels = true answer (1 = fake)
preds = (probs >= THRESHOLD).astype(int)          # the model's answer (1 = fake)

real = labels == 0
fake = labels == 1
print(f"Checked {len(labels):,} test photos.")
print(f"Real faces: {real.sum():,}  -> let through {(real & (preds == 0)).sum():,}, wrongly flagged {(real & (preds == 1)).sum():,}")
print(f"Fake faces: {fake.sum():,}  -> caught {(fake & (preds == 1)).sum():,}, missed {(fake & (preds == 0)).sum():,}")
print(f"Accuracy: {(preds == labels).mean():.3f}")

## Step 4 — Find the worst mistakes

**What this does:** collects the two kinds of mistake and sorts them so the **most confident** mistakes come first:

- worst **false accepts**: fake faces with the **lowest** `P(fake)` (the model was very sure they were real);
- worst **false rejects**: real faces with the **highest** `P(fake)` (the model was very sure they were fake).

In [ ]:
false_accepts = [i for i in range(len(labels)) if labels[i] == 1 and preds[i] == 0]   # fake called real
false_rejects = [i for i in range(len(labels)) if labels[i] == 0 and preds[i] == 1]   # real called fake

false_accepts.sort(key=lambda i: probs[i])      # lowest P(fake) first
false_rejects.sort(key=lambda i: -probs[i])     # highest P(fake) first

print(f"False accepts (fake -> real): {len(false_accepts)}")
print(f"False rejects (real -> fake): {len(false_rejects)}")

## Step 5 — Look at the mistakes with Grad-CAM heatmaps

**What this does:** for each of the worst mistakes, shows the photo (left) and the same photo with the heatmap on
top (right). **Red/yellow** = the area that pushed the model towards "fake"; **blue** = little influence.

**What to look for:**
- Is the red on the **face** (eyes, mouth, skin, hair edges)? That is where AI artefacts usually are.
- Is the red on the **background** or the **image border**? Then the model may be using a shortcut.
- Do the mistakes have something in common (glasses, smiles, dark photos, children, …)?

**Caution:** a heatmap is a rough hint, not proof. It is low-resolution and can look convincing even when it is wrong.

Change `NUM_TO_SHOW` to see more or fewer of each kind.

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import BinaryClassifierOutputTarget

from deepfake_kyc.models import gradcam_target_layer

NUM_TO_SHOW = 4        # how many of each kind of mistake

# Grad-CAM looks at the model's last convolution layer to see which areas mattered most.
cam = GradCAM(model=model, target_layers=[gradcam_target_layer(model)])

def heatmap(image):
    """Return (P(fake), photo with the heatmap drawn on top)."""
    image = image.convert("RGB")
    x = transform(image).unsqueeze(0).to(device)               # prepare the photo like in Step 3
    with torch.no_grad():
        p_fake = torch.sigmoid(model(x)).item()
    heat = cam(input_tensor=x, targets=[BinaryClassifierOutputTarget(1)])[0]   # 1 = "fake"
    photo = np.asarray(image.resize((IMAGE_SIZE, IMAGE_SIZE)), dtype=np.float32) / 255
    return p_fake, show_cam_on_image(photo, heat, use_rgb=True)

to_show = [(i, "false accept: fake called real") for i in false_accepts[:NUM_TO_SHOW]] + \
          [(i, "false reject: real called fake") for i in false_rejects[:NUM_TO_SHOW]]

if to_show:
    fig, axes = plt.subplots(len(to_show), 2, figsize=(6, 3 * len(to_show)), squeeze=False)
    for row, (i, kind) in zip(axes, to_show):
        image = Image.open(paths[i])
        p_fake, overlay = heatmap(image)
        row[0].imshow(image.convert("RGB"))
        row[0].set_title(kind, fontsize=9)
        row[1].imshow(overlay)
        row[1].set_title(f"P(fake) = {p_fake:.2f}", fontsize=9)
        row[0].axis("off")
        row[1].axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No mistakes to show: the model got every checked photo right.")

## Step 6 (try it) — Does damage change where the model looks?

**What this does:** takes one test photo and shows the heatmap on the clean photo and on damaged versions of it,
using the same damage code as notebook 02. If the red area jumps around or the `P(fake)` changes a lot,
that damage confuses the model. Models trained on clean + damaged photos should change less than the
`06_effnet_clean` control.

Change `PHOTO`, `DAMAGE` and run the cell again.

In [ ]:
from deepfake_kyc.degradations import DEGRADATIONS

PHOTO = 0              # which photo: a number from 0 to the number of checked photos - 1
DAMAGE = "blur"        # try "jpeg", "blur", "downscale", "noise" or "screen"

image = Image.open(paths[PHOTO]).convert("RGB")
truth = "fake" if labels[PHOTO] == 1 else "real"
versions = [("clean", image)] + [(f"{DAMAGE} {s}", DEGRADATIONS[DAMAGE](s)(image)) for s in [1, 3, 5]]

fig, axes = plt.subplots(2, len(versions), figsize=(3 * len(versions), 6))
for col, (name, version) in enumerate(versions):
    p_fake, overlay = heatmap(version)
    axes[0, col].imshow(version)
    axes[0, col].set_title(name)
    axes[1, col].imshow(overlay)
    axes[1, col].set_title(f"P(fake) = {p_fake:.2f}")
    axes[0, col].axis("off")
    axes[1, col].axis("off")
fig.suptitle(f"One {truth} test photo: clean vs. {DAMAGE} at severity 1, 3, 5")
plt.tight_layout()
plt.show()

## Next

- Compare these mistakes with another model: change `MODEL` in Step 2 (for example `"03_effnet_finetune"` vs.
  the `"06_effnet_clean"` control) and run Steps 2–6 again.
- The overall scores for every model are in notebook **03_results_charts**.

## Quick notes to remember

- This notebook only **reads** files; it saves nothing.
- A false accept costs fraud; a false reject costs a customer. In the report, show a few of each.
- Grad-CAM is a rough guide, not proof. Say this in the report.